# Fine-tuning Models for Sentiment Analysis

This notebook demonstrates how to fine-tune a Hugging Face model for sentiment analysis using Amazon SageMaker.

We'll use the GLUE SST-2 dataset to fine-tune a DistilBERT model for binary sentiment classification.

## Setup

First, let's install the required packages:

In [ ]:
!pip install -U pip
!pip install sagemaker boto3 datasets transformers scikit-learn

Import the necessary libraries:

In [ ]:
import os
import json
import time
import logging
import boto3
import sagemaker
from sagemaker.huggingface import HuggingFace
from sagemaker.pytorch import PyTorchModel
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer
from datasets import load_dataset
from transformers import AutoTokenizer

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

## Configuration

Set up the configuration for the fine-tuning job:

In [ ]:
# Get the default S3 bucket for this SageMaker session
session = sagemaker.Session()
S3_BUCKET = session.default_bucket()
S3_PREFIX = "sentiment-analysis"

print(f"Using S3 bucket: {S3_BUCKET}")

# S3 paths
OUTPUT_PATH = f"s3://{S3_BUCKET}/{S3_PREFIX}/output"

# Model and training parameters
MODEL_ID = "distilbert-base-uncased"
DATASET_NAME = "glue"
DATASET_CONFIG = "sst2"
INSTANCE_TYPE = "ml.g5.2xlarge"  # Single GPU instance (cost-optimized for our workload)
INSTANCE_COUNT = 1
EPOCHS = 3
BATCH_SIZE = 32
LEARNING_RATE = 5e-5
MAX_LENGTH = 128

print(f"\n💡 Instance Selection Note:")
print(f"   We're using {INSTANCE_TYPE} because our training script is designed for single-GPU training.")
print(f"   This instance provides excellent price/performance for our DistilBERT + SST-2 workload.")
print(f"   Multi-GPU instances like ml.g5.12xlarge would be under-utilized and more expensive.")

## Prepare and Upload Dataset

Let's prepare the SST2 dataset (from GLUE) and convert it to the proper format for SageMaker:

In [ ]:
def prepare_dataset(dataset_name, dataset_config, model_id, max_length=128):
    """Load, tokenize, and prepare dataset for SageMaker"""
    logger.info(f"Loading dataset: {dataset_name}/{dataset_config}")
    dataset = load_dataset(dataset_name, dataset_config)
    
    logger.info(f"Loading tokenizer for model: {model_id}")
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    
    logger.info("Tokenizing dataset")
    tokenized_datasets = {}
    
    # Process training split
    if "train" in dataset:
        def tokenize_function(examples):
            tokenized = tokenizer(
                examples["sentence"],
                padding="max_length",
                truncation=True,
                max_length=max_length,
            )
            # Keep the labels
            tokenized["labels"] = examples["label"]
            return tokenized
        
        tokenized_datasets["train"] = dataset["train"].map(
            tokenize_function,
            batched=True,
            remove_columns=["sentence"],  # Only remove sentence column, keep label
        )
    
    # Process validation split
    if "validation" in dataset:
        def tokenize_function_val(examples):
            tokenized = tokenizer(
                examples["sentence"],
                padding="max_length",
                truncation=True,
                max_length=max_length,
            )
            # Keep the labels
            tokenized["labels"] = examples["label"]
            return tokenized
        
        tokenized_datasets["validation"] = dataset["validation"].map(
            tokenize_function_val,
            batched=True,
            remove_columns=["sentence"],  # Only remove sentence column, keep label
        )
    
    return tokenized_datasets

In [ ]:
def upload_to_s3(dataset_dict, s3_bucket, s3_prefix):
    """Upload datasets to S3"""
    logger.info("Saving datasets locally")
    os.makedirs("data", exist_ok=True)
    
    s3_paths = {}
    for split_name, dataset in dataset_dict.items():
        local_path = f"data/{split_name}"
        dataset.save_to_disk(local_path)
        
        logger.info(f"Uploading {split_name} dataset to S3")
        s3_path = f"s3://{s3_bucket}/{s3_prefix}/data/{split_name}"
        
        # Use AWS CLI for uploading
        os.system(f"aws s3 cp {local_path} {s3_path} --recursive")
        
        logger.info(f"Dataset uploaded to {s3_path}")
        s3_paths[split_name] = s3_path
    
    return s3_paths

In [ ]:
# Prepare and upload the dataset
tokenized_datasets = prepare_dataset(DATASET_NAME, DATASET_CONFIG, MODEL_ID, MAX_LENGTH)
s3_paths = upload_to_s3(tokenized_datasets, S3_BUCKET, S3_PREFIX)
print(f"Dataset uploaded successfully: {s3_paths}")

## Training Script

The training script is available as a separate file:

In [ ]:
# The training script is available as a separate file: scripts/train.py
# This script contains all the necessary components for fine-tuning:
# - Dataset loading with proper error handling
# - compute_metrics function for evaluation
# - Argument parsing for SageMaker environment variables
# - Main training loop using Hugging Face Trainer

print("Training script: scripts/train.py")

## Fine-tune the Model

Now let's fine-tune the model using SageMaker with the latest supported versions:

In [ ]:
# Get execution role
role = sagemaker.get_execution_role()
logger.info(f"Using SageMaker execution role: {role}")

In [ ]:
# Define hyperparameters
hyperparameters = {
    'epochs': EPOCHS,
    'train_batch_size': BATCH_SIZE,
    'eval_batch_size': BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'model_name': MODEL_ID,
}

# Define metric definitions for tracking
metric_definitions = [
    {'Name': 'train:loss', 'Regex': 'train_loss: ([0-9\\.]+)'},
    {'Name': 'eval:loss', 'Regex': 'eval_loss: ([0-9\\.]+)'},
    {'Name': 'eval:accuracy', 'Regex': 'eval_accuracy: ([0-9\\.]+)'},
    {'Name': 'eval:f1', 'Regex': 'eval_f1: ([0-9\\.]+)'},
]

print(f"Hyperparameters: {hyperparameters}")
print(f"Output path: {OUTPUT_PATH}")

In [ ]:
# Create Hugging Face estimator with latest supported versions
logger.info("Creating HuggingFace estimator...")
huggingface_estimator = HuggingFace(
    entry_point='train.py',
    source_dir='./scripts',
    instance_type=INSTANCE_TYPE,
    instance_count=INSTANCE_COUNT,
    role=role,
    transformers_version='4.49.0',  # Latest supported version
    pytorch_version='2.5.1',        # Latest supported version
    py_version='py311',             # Modern Python version
    hyperparameters=hyperparameters,
    metric_definitions=metric_definitions,
    output_path=OUTPUT_PATH
)

print("HuggingFace estimator created successfully!")
print(f"Using instance: {INSTANCE_TYPE} (single GPU, cost-optimized)")

In [ ]:
# Define data channels
train_data_path = s3_paths['train']
validation_data_path = s3_paths['validation']

data_channels = {
    'train': train_data_path,
    'validation': validation_data_path
}

print(f"Data channels: {data_channels}")

In [ ]:
# Start training job
logger.info("Starting training job...")
huggingface_estimator.fit(data_channels, wait=True)
logger.info(f"Training job completed. Model artifacts saved to: {huggingface_estimator.model_data}")
print(f"\n✅ Training completed successfully!")
print(f"Model artifacts: {huggingface_estimator.model_data}")

## Deploy the Model using PyTorch Container

Following the pattern from other notebooks, we'll deploy using a PyTorch container instead of HuggingFace container for better version compatibility:

In [ ]:
# The inference script is available as a separate file: scripts/fine_tuning_inference.py
# This script contains all the necessary components for inference:
# - model_fn: Load the fine-tuned model and tokenizer
# - input_fn: Parse and handle different input formats
# - predict_fn: Run inference and return sentiment predictions
# - output_fn: Format the output as JSON

print("Inference script: scripts/fine_tuning_inference.py")

In [ ]:
# Create PyTorch model for deployment
pytorch_model = PyTorchModel(
    model_data=huggingface_estimator.model_data,
    role=role,
    framework_version="2.5.1",  # Supported PyTorch version
    py_version="py311",         # Modern Python version
    entry_point="fine_tuning_inference.py",
    source_dir="./scripts",
    env={
        'HF_TASK': 'text-classification'
    }
)

print("PyTorch model created for deployment!")

In [ ]:
# Deploy model to endpoint
endpoint_name = f"sentiment-analysis-{int(time.time())}"
predictor = pytorch_model.deploy(
    initial_instance_count=1,
    instance_type='ml.m5.xlarge',
    endpoint_name=endpoint_name
)
print(f"Model deployed to endpoint: {endpoint_name}")

## Test the Model (Optional)

If you deployed the model, you can test it with some sample text:

In [ ]:
# Test the model
# Set serializers
predictor.serializer = JSONSerializer()
predictor.deserializer = JSONDeserializer()

# Test samples
sample_texts = [
    "This movie was fantastic! I really enjoyed it.",
    "This movie was terrible. I hated it.",
    "The movie was okay, nothing special.",
    "Amazing performance by the actors!",
    "Boring and predictable plot."
]

print("\n" + "="*60)
print("MODEL TESTING RESULTS")
print("="*60)

for i, text in enumerate(sample_texts, 1):
    start_time = time.time()
    response = predictor.predict({"inputs": text})
    end_time = time.time()
    
    print(f"\nTest {i}:")
    print(f"Text: {text}")
    print(f"Prediction: {response['label']} (confidence: {response['score']:.4f})")
    print(f"Scores: NEGATIVE={response['scores']['NEGATIVE']:.4f}, POSITIVE={response['scores']['POSITIVE']:.4f}")
    print(f"Inference time: {(end_time - start_time) * 1000:.2f} ms")

print("\n" + "="*60)

## Clean Up (Optional)

Don't forget to delete the endpoint when you're done to avoid incurring charges:

In [ ]:
# Delete the endpoint (uncomment if you deployed the model)
# predictor.delete_endpoint()
# print("Endpoint deleted successfully!")

## Summary

In this notebook, we demonstrated how to:

1. **Prepare the dataset**: Load the GLUE SST-2 dataset, tokenize it, and save in the proper format
2. **Upload to S3**: Store the processed dataset in S3 for SageMaker training
3. **Create a training script**: Write a robust training script using Hugging Face Transformers
4. **Fine-tune the model**: Use SageMaker's HuggingFace estimator to fine-tune DistilBERT
5. **Monitor training**: Track the training job status and metrics
6. **Deploy with PyTorch**: Deploy the trained model using PyTorch container for better compatibility
7. **Test**: Test the deployed model with sample text

The fine-tuned model can now classify text as positive or negative sentiment with improved accuracy on your specific domain.

**Key Technical Decisions:**
- **Training**: Uses latest HuggingFace versions (4.49.0/2.5.1/py311) for training
- **Instance Selection**: Uses ml.g5.2xlarge for cost-effective single-GPU training (our script doesn't use multi-GPU)
- **Deployment**: Uses PyTorch container instead of HuggingFace container for better version compatibility
- **Inference**: Custom inference script provides full control over model loading and prediction logic

This approach provides the best of both worlds: modern versions for training and reliable deployment infrastructure, while being cost-optimized for our specific workload characteristics.